In [1]:
import pandas as pd
df = pd.read_csv("feature_eng_df_final.csv")

In [2]:
df.head()

,age,education,monthly_salary,years_of_employment,company_type,house_type,monthly_rent,family_size,dependents,school_fees,...,affordability_ratio,credit_score_category,credit_score_numeric,combined_credit_risk,employment_tenure_category,is_long_term_employed,income_per_family_member,savings_to_income_ratio,credit_stability_score,loan_affordability_index
0,38.0,3.0,11.321777,0.641854,2.0,0.0,1.280128,3,2,0.0,...,-2.443993,Fair,2,3,Entry-level,0,3.773926,0.042580,423.623565,1.205905
1,38.0,1.0,9.975855,2.079442,4.0,1.0,-0.814566,2,1,5100.0,...,-513.331366,Good,1,2,Mid-level,1,4.987927,-0.177067,1484.721261,1.178826
2,38.0,3.0,11.363276,1.916923,0.0,2.0,-0.814566,4,3,0.0,...,-1.321418,Fair,2,2,Mid-level,1,2.840819,0.346565,1245.999698,1.111593
3,58.0,0.0,11.109473,1.163151,2.0,2.0,-0.814566,5,4,11400.0,...,-1027.461671,Good,1,1,Entry-level,0,2.221895,0.200884,796.758305,1.136398
4,48.0,3.0,10.956073,1.481605,2.0,1.0,-0.814566,4,3,9400.0,...,-859.501890,Excellent,0,0,Mid-level,0,2.739018,-0.153537,1140.835497,1.135187


In [3]:
# dropping credit score category since it already encoded as new column credit_score_numeric
df.drop('credit_score_category',axis=1,inplace=True)

In [4]:
#let's encode the binned age group column derived from age column in to ordinal encoder
df['age_group'].unique()
age_group_order = ['25-34','35-44', '55-64', '45-54']
from sklearn.preprocessing import OrdinalEncoder
age_group_encoder = OrdinalEncoder(categories=[age_group_order])
df['age_group'] = age_group_encoder.fit_transform(df[['age_group']])
print("after encoding df['age_group']",df['age_group'].unique())
df.drop('age',axis=1,inplace=True)

after encoding df['age_group'] [1. 2. 3. 0.]


In [5]:
#let encode binned employment experience type from year_of_employment column 
df['employment_tenure_category'].unique()
employment_order = ['Entry-level', 'Mid-level', 'Experienced']
emp_category_encoder = OrdinalEncoder(categories=[employment_order])
df['employment_tenure_category'] = emp_category_encoder.fit_transform(df[['employment_tenure_category']])
print("after encoding emp tenure category ",df['employment_tenure_category'].unique())
df.drop('years_of_employment',axis=1,inplace=True)

after encoding emp tenure category  [0. 1. 2.]


In [6]:
from sklearn.model_selection import train_test_split
X = df.drop(['max_monthly_emi','emi_eligibility'],axis=1)
y= df['emi_eligibility']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("x train shape",X_train.shape)
print("X test shape",X_test.shape)
print("y train shape",y_train.shape)
print("y test shape",y_test.shape)

x train shape (322243, 40)
X test shape (80561, 40)
y train shape (322243,)
y test shape (80561,)


In [7]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix,f1_score,roc_auc_score

In [8]:
y_train.value_counts()

emi_eligibility
2.0    248697
0.0     59594
1.0     13952
Name: count, dtype: int64

In [9]:
xgb_model_1 = XGBClassifier(n_estimators = 200,max_depth=4,learning_rate=0.1,eval_metric="logloss",random_state=42,use_label_encoder=False)

In [10]:
xgb_model_1.fit(X_train,y_train)

C:\Users\RAM\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:12:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [11]:
y_xb_predict = xgb_model_1.predict(X_test)

In [19]:
xb_accuracy = accuracy_score(y_test,y_xb_predict)
print("accuracy--",xb_accuracy)
xb_precision = precision_score(y_test,y_xb_predict,average="micro")
print("precision---",xb_precision)
xb_recall = recall_score(y_test,y_xb_predict,average=None)
print("recall---",xb_recall)
xb_f1_score = f1_score(y_test,y_xb_predict,average=None)
print("f1_score",xb_f1_score)

accuracy-- 0.9393875448417969
precision--- 0.9393875448417969
recall--- [0.93484263 0.02729384 0.99059541]
f1_score [0.91015389 0.05161999 0.97151537]


In [17]:
xb_confusion = confusion_matrix(y_test,y_xb_predict)
print("confusion matrix")
print(xb_confusion)

confusion matrix
[[13544    82   862]
 [ 1163    94  2187]
 [  567    22 62040]]


In [16]:
# Data has more imbalance let's try different approach


In [21]:
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)



,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [23]:
xgb2_y_predict = xgb_model.predict(X_test)

In [25]:
xb_confusion_2 = confusion_matrix(y_test,xgb2_y_predict)
print("confusion matrix")
print(xb_confusion_2)

confusion matrix
[[12317  2115    56]
 [  383  2970    91]
 [  444  6029 56156]]


In [27]:
xb_accuracy_2 = accuracy_score(y_test,xgb2_y_predict)
print("accuracy--",xb_accuracy_2)
xb_precision_2 = precision_score(y_test,xgb2_y_predict,average="macro")
print("precision---",xb_precision_2)
xb_recall_2 = recall_score(y_test,xgb2_y_predict,average="macro")
print("recall---",xb_recall_2)
xb_f1_score_2 = f1_score(y_test,xgb2_y_predict,average="macro")
print("f1_score",xb_f1_score_2)

accuracy-- 0.8868186839786001
precision--- 0.733900401626561
recall--- 0.8697221706123296
f1_score 0.7479545422587689


In [34]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1]
}

grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
